# Checkpoint 03 - tempo de aquisição, buffer e perdas

Mantemos o cliente SSE do **Checkpoint 02**.

A novidade é reconstruir corretamente o tempo do sinal a partir dos índices de amostra, manter uma janela em memória e representar perdas de dados sem inventar valores.

## 1. Configuração

In [ ]:
# CONFIGURAÇÃO DA OFICINA
# Use a mesma URL do Firebase Realtime Database fornecida para a oficina.
FIREBASE_URL = "https://oficina-python-saude-teste-default-rtdb.firebaseio.com"  # Endereço base do Realtime Database usado na oficina.
STREAM_PATH = "oficina/live"  # Caminho do nó que contém sempre o pacote mais recente.

# Parâmetros da oficina
FS = 250  # Frequência de amostragem dos sinais: 250 amostras por segundo.
AMOSTRAS_POR_PACOTE = 50  # Quantidade esperada de amostras em cada pacote publicado.
JANELA_S = 8  # Duração, em segundos, da janela de sinal mantida no buffer.

## 2. Reutilização dos checkpoints anteriores

Este bloco reúne a comunicação REST/SSE já apresentada e recupera `validar_pacote()`, introduzida no Checkpoint 01.

Você pode executar o bloco completo sem precisar explicar novamente cada função.

In [ ]:
import json                         # Converte as mensagens JSON do SSE em dicionários Python.
import time                         # Controla a duração da captura do fluxo.
import numpy as np                  # Manipula os sinais e representa amostras ausentes com np.nan.
import requests                     # Abre a conexão HTTP/SSE com o Firebase.

# Função: Monta a URL REST completa do nó do Firebase, evitando barras duplicadas.
def firebase_node_url(firebase_url, node=STREAM_PATH):
    base = firebase_url.rstrip("/")  # Remove uma barra final para evitar "//" ao montar a URL.
    node = node.strip("/")  # Remove barras no início/fim do caminho antes de juntá-lo à URL.
    return f"{base}/{node}.json"  # O sufixo .json permite acessar esse nó pela API REST do Firebase.

# Função: Faz uma leitura única do pacote atualmente armazenado no nó da oficina.
def ler_pacote_atual(firebase_url=FIREBASE_URL, node=STREAM_PATH):
    r = requests.get(firebase_node_url(firebase_url, node), timeout=10)  # Faz um GET e espera no máximo 10 s pela resposta.
    r.raise_for_status()  # Interrompe com erro se o servidor responder com código HTTP de falha.
    return r.json()  # Converte a resposta JSON do Firebase diretamente em um objeto Python.


# Função: Separa o fluxo SSE em eventos completos, identificando os campos event e data.
def iterar_sse(response):
    """Interpreta eventos Server-Sent Events retornados pelo Firebase REST."""
    evento = None
    dados = []

    for raw in response.iter_lines(decode_unicode=True):  # Lê a resposta contínua linha a linha, já convertida para texto.
        if raw is None:
            continue

        linha = raw.strip()  # Remove espaços/quebras de linha antes de interpretar o campo SSE.

        if linha == "":  # No SSE, uma linha vazia indica o fim de um evento.
            if evento is not None and dados:
                yield evento, "\n".join(dados)
            evento = None
            dados = []
            continue

        if linha.startswith(":"):  # Linhas iniciadas por ":" são comentários/keep-alive do protocolo SSE.
            continue

        if linha.startswith("event:"):
            evento = linha.split(":", 1)[1].strip()  # Extrai apenas o valor que vem depois de "event:".
        elif linha.startswith("data:"):
            dados.append(linha.split(":", 1)[1].lstrip())  # Guarda apenas o conteúdo que vem depois de "data:".

# Função: Mantém a conexão SSE aberta e entrega, um a um, os pacotes válidos recebidos.
def pacotes_firebase(firebase_url=FIREBASE_URL, node=STREAM_PATH, duracao_s=20):
    """Gera pacotes recebidos pelo stream SSE do Firebase."""
    url = firebase_node_url(firebase_url, node)  # Obtém a URL REST completa do nó que será acompanhado.
    headers = {"Accept": "text/event-stream"}  # Informa ao Firebase que queremos receber Server-Sent Events.
    inicio = time.time()  # Marca o instante inicial para limitar a duração da captura.

    with requests.get(url, headers=headers, stream=True, timeout=(10, 35)) as response:
        response.raise_for_status()  # Garante que a conexão foi aceita antes de processar o fluxo.
        for evento, texto in iterar_sse(response):
            if time.time() - inicio >= duracao_s:
                return
            if evento not in ("put", "patch"):
                continue

            envelope = json.loads(texto)  # Transforma o texto JSON do evento SSE em dicionário Python.
            data = envelope.get("data")  # No SSE do Firebase, o conteúdo alterado vem dentro da chave data.
            if isinstance(data, dict) and "seq" in data:
                yield data  # Entrega o pacote ao laço for sem encerrar a conexão SSE.

# Função: Confere os campos e tamanhos do pacote e converte os sinais para vetores NumPy.
def validar_pacote(pacote):
    campos = (
        "session_id", "seq", "sample_start", "fs", "n",
        "ecg", "ppg_ir", "ppg_red"
    )
    faltando = [k for k in campos if k not in pacote]  # Lista quais campos obrigatórios não chegaram no pacote.
    if faltando:
        raise ValueError(f"campos ausentes: {faltando}")

    n = int(pacote["n"])  # Converte o tamanho informado para inteiro antes de comparar com os vetores.
    ecg = np.asarray(pacote["ecg"], dtype=float)  # Converte a lista de ECG recebida em vetor numérico.
    ir = np.asarray(pacote["ppg_ir"], dtype=float)  # Converte o canal PPG infravermelho em vetor numérico.
    red = np.asarray(pacote["ppg_red"], dtype=float)  # Converte o canal PPG vermelho em vetor numérico.

    if not (len(ecg) == len(ir) == len(red) == n):
        raise ValueError("vetores com tamanhos incompatíveis")

    return {
        "session_id": str(pacote["session_id"]),
        "seq": int(pacote["seq"]),
        "sample_start": int(pacote["sample_start"]),
        "fs": float(pacote["fs"]),
        "n": n,
        "timestamp_pub": float(pacote.get("timestamp_pub", 0.0)),
        "ecg": ecg,
        "ppg_ir": ir,
        "ppg_red": red,
    }

## 3. Novidade deste checkpoint — buffers, tempo de aquisição e lacunas

Agora acrescentamos `deque` e criamos buffers para:

- índice absoluto da amostra;
- ECG;
- PPG infravermelho;
- PPG vermelho.

Quando há uma lacuna entre pacotes, inserimos `NaN`. Isso preserva a posição temporal da amostra ausente sem inventar um valor.

In [ ]:
from collections import deque       # Cria buffers eficientes, com inclusão e remoção rápida nas extremidades.

sample_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
ecg_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
ir_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
red_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.

active_session = None
last_sample_end = None
missing_samples = 0

# Função: Insere um pacote nos buffers, marca perdas com NaN e mantém apenas a janela recente.
def inserir_pacote(p):
    global active_session, last_sample_end, missing_samples  # Permite que a função atualize essas variáveis criadas fora dela.

    if active_session != p["session_id"]:
        active_session = p["session_id"]
        last_sample_end = None
        missing_samples = 0
        sample_buf.clear()  # Remove os índices da sessão anterior.
        ecg_buf.clear()  # Remove as amostras de ECG da sessão anterior.
        ir_buf.clear()  # Remove as amostras PPG infravermelhas da sessão anterior.
        red_buf.clear()  # Remove as amostras PPG vermelhas da sessão anterior.
        print("Nova sessão - buffers reiniciados.")

    # Só é possível procurar uma lacuna se já recebemos um pacote anterior.
    if last_sample_end is not None:
        # A próxima amostra esperada deve vir imediatamente após a última amostra recebida.
        esperado = last_sample_end + 1

        # Se o novo pacote começa depois do índice esperado, houve perda de amostras entre os pacotes.
        if p["sample_start"] > esperado:
            # Quantidade de amostras que ficaram faltando entre o pacote anterior e o atual.
            gap = p["sample_start"] - esperado

            # Acumula o número total de amostras perdidas detectadas durante a execução.
            missing_samples += gap

            # Percorre todos os índices correspondentes às amostras que não chegaram.
            for sample_idx in range(esperado, p["sample_start"]):
                # Guarda o índice da amostra ausente para preservar a continuidade do eixo temporal.
                sample_buf.append(sample_idx)

                # np.nan marca uma amostra ausente sem inventar um valor de ECG.
                ecg_buf.append(np.nan)

                # Marca como ausente a amostra correspondente do PPG infravermelho.
                ir_buf.append(np.nan)

                # Marca como ausente a amostra correspondente do PPG vermelho.
                red_buf.append(np.nan)

    # Percorre todas as amostras presentes no pacote atual.
    for i in range(p["n"]):
        # Calcula o índice absoluto da amostra no fluxo contínuo.
        sample_buf.append(p["sample_start"] + i)

        # Converte a amostra de ECG para float e a acrescenta ao buffer.
        ecg_buf.append(float(p["ecg"][i]))

        # Converte a amostra do PPG infravermelho para float e a acrescenta ao buffer.
        ir_buf.append(float(p["ppg_ir"][i]))

        # Converte a amostra do PPG vermelho para float e a acrescenta ao buffer.
        red_buf.append(float(p["ppg_red"][i]))

    last_sample_end = p["sample_start"] + p["n"] - 1  # Guarda o índice da última amostra válida deste pacote.

    # conserva somente os últimos JANELA_S segundos do relógio do sinal
    minimo = last_sample_end - int(JANELA_S * p["fs"])  # Calcula o índice mínimo que ainda pertence à janela temporal.
    while sample_buf and sample_buf[0] < minimo:
        sample_buf.popleft()  # Remove o índice mais antigo.
        ecg_buf.popleft()  # Remove a amostra de ECG correspondente.
        ir_buf.popleft()  # Remove a amostra PPG IR correspondente.
        red_buf.popleft()  # Remove a amostra PPG Red correspondente.

## 4. Executar e observar o buffer

O laço usa o mesmo stream SSE, valida cada pacote e chama apenas a nova lógica de inserção no buffer.

In [ ]:
for pacote in pacotes_firebase(duracao_s=20):
    p = validar_pacote(pacote)
    inserir_pacote(p)
    duracao = (sample_buf[-1] - sample_buf[0]) / p["fs"] if len(sample_buf) > 1 else 0  # Converte a extensão do buffer, em amostras, para segundos.
    print(
        f"seq={p['seq']:4d}  buffer={len(sample_buf):4d}  "
        f"janela={duracao:4.1f}s  missing={missing_samples}"
    )

print("CHECKPOINT 3 OK")